# 08 — Avaliação de retreino e explicabilidade (XAI)

**Decisão de negócio da DESLOC:** saber se há evidência para substituir ou retreinar o modelo que informa o preço **antes** da corrida. Este notebook complementa as aulas espelhadas em `01`–`06`; ele não altera o código do professor.

**Resultado possível com os dados atuais:** caracterizar as 104 corridas, testar a estabilidade do KNN em validação cruzada e explicar seus resultados na amostra. **Ainda não é possível concluir que o modelo de produção degradou**, comparar com ele ou medir drift sem previsões históricas, base original de treino e horário de solicitação.

**Fontes atuais:** `desloc.gold.transacoes_eda` e `desloc.gold.transacoes_ml`. Valores pagos são receita bruta por corrida, não lucro. A distância `distancia_lalo` é geográfica em linha reta, não quilometragem percorrida.

## 1. Como este notebook cobre o CRISP-DM

| Fase | Evidência no projeto | Lacuna para decisão real |
|---|---|---|
| Entendimento do negócio | PDF do modelo de negócios e critérios abaixo | Meta de erro e impacto em conversão aprovados pelo negócio |
| Entendimento dos dados | Bronze, Silver, Gold e `04_eda` | Base original de treino, dados recentes identificados, solicitações canceladas |
| Preparação | `05_preparacao_dados` | Data/hora disponível antes da corrida, identificador da corrida |
| Modelagem | KNN do `06_ml` | Comparação justa com modelo atual e avaliação futura |
| Avaliação | MAE/RMSE, validação cruzada e XAI deste notebook | Erro e viés do modelo atual no mesmo conjunto |
| Implantação | Plano e critérios nesta seção | Validação com responsáveis, custo, rollout e monitoramento |

O CRISP-DM é iterativo: se a coleta ou a definição de features mudar, a avaliação deve voltar às fases anteriores.

## 2. Importações e parâmetros
A validação cruzada abaixo é **diagnóstica**: a tabela ML contém features derivadas da hora de entrega. Ela não mede desempenho de uma cotação feita antes da corrida.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

TABELA_EDA = "desloc.gold.transacoes_eda"
TABELA_ML = "desloc.gold.transacoes_ml"
ALVO = "vlr_pago"
FEATURES = ["dia", "dia_semana", "mes", "hora", "distancia_lalo",
            "latitude_origem", "longitude_origem",
            "latitude_destino", "longitude_destino"]
SEMENTE = 42
TESTE = 0.20

## 3. Reconciliar a amostra com o diagnóstico anterior
Na extração fornecida, foram observadas **104 corridas**, valor pago total de **R$ 2.373,04**, preço médio de **R$ 22,82**, correlação entre distância geográfica e preço de cerca de **0,967**, segunda-feira com **25/104 corridas (24,0%)** e período da manhã com **47/104 (45,2%)**. Esses números são referência da amostra, não uma regra permanente. A célula os recalcula nas tabelas Gold.

In [ ]:
eda = spark.table(TABELA_EDA).toPandas()
ml = spark.table(TABELA_ML).toPandas()
if eda.empty or ml.empty:
    raise ValueError("Uma das tabelas Gold está vazia")
for coluna in [ALVO, "distancia_lalo", "dia_semana", "periodo"]:
    eda[coluna] = pd.to_numeric(eda[coluna], errors="coerce")
for coluna in [ALVO, "periodo", *FEATURES]:
    ml[coluna] = pd.to_numeric(ml[coluna], errors="coerce")
resumo = pd.DataFrame([{
    "corridas_eda": len(eda),
    "corridas_ml": len(ml),
    "valor_pago_total_R$": eda[ALVO].sum(),
    "preco_medio_R$": eda[ALVO].mean(),
    "correlacao_distancia_preco": eda[["distancia_lalo", ALVO]].corr().iloc[0, 1],
    "segunda_corridas_pct": 100 * (eda.dia_semana == 2).mean(),
    "manha_corridas_pct": 100 * (eda.periodo == 1).mean(),
    "nulos_preco_eda": int(eda[ALVO].isna().sum()),
    "nulos_distancia_eda": int(eda.distancia_lalo.isna().sum()),
}])
display(resumo.round(3))
if len(eda) != len(ml):
    print("ATENÇÃO: EDA e ML têm contagens diferentes; conferir origem e filtros antes de comparar.")

## 4. Critérios de aceite para a decisão de retreino

| Critério | Regra objetiva | Situação com os dados atuais |
|---|---|---|
| Qualidade | Alvo e features sem nulos ou valores impossíveis; mesma população nas comparações | Verificável nas Gold |
| Disponibilidade temporal | Toda entrada deve existir no momento da cotação | **Não atende:** calendário vem da entrega |
| Comparabilidade | Modelo atual e candidato avaliados nas **mesmas corridas futuras**, com mesmo alvo e preço em reais | **Pendente:** faltam previsões do atual |
| Ganho de erro | MAE do candidato menor que o atual; RMSE não deve piorar materialmente | **Pendente:** limiar de ganho deve ser acordado com o negócio |
| Segmentos | Inspecionar erro por distância, período e região; evitar regressão importante em um grupo | **Pendente:** amostras suficientes por grupo |
| Drift | Comparar dados recentes com a base original e/ou janelas sucessivas | **Pendente:** falta baseline identificada |
| Impacto | Medir aceite, cancelamento, reclamação e margem antes/depois | **Pendente:** a base só tem corridas finalizadas e preço pago |
| XAI | Explicações globais e locais coerentes, sem indicar causalidade inexistente | Diagnóstico possível abaixo |

Os limiares de melhoria e de piora por segmento precisam ser definidos com o professor ou responsáveis de negócio **antes** de olhar o teste final. Não fixamos um percentual arbitrário como se fosse critério aprovado.

## 5. Verificar qualidade e risco de vazamento temporal
`duracao` foi excluída da tabela ML, mas as features `dia`, `dia_semana`, `mes` e `hora` ainda derivam de `datahora_entregue`. O horário de entrega ocorre depois do momento em que o passageiro precisa ver o preço. Mesmo um MAE baixo com essa tabela não valida o uso em produção.

In [ ]:
nulos = ml[[ALVO, "periodo", *FEATURES]].isna().sum().rename("nulos").to_frame()
faixas = pd.DataFrame({
    "checagem": ["preco_nao_positivo", "distancia_nao_positiva", "hora_fora_0_23",
                "dia_semana_fora_1_7", "coordenadas_fora_da_faixa"],
    "linhas": [int((ml[ALVO] <= 0).sum()), int((ml.distancia_lalo <= 0).sum()),
               int((~ml.hora.between(0, 23)).sum()),
               int((~ml.dia_semana.between(1, 7)).sum()),
               int((~ml.latitude_origem.between(-90, 90) |
                    ~ml.latitude_destino.between(-90, 90) |
                    ~ml.longitude_origem.between(-180, 180) |
                    ~ml.longitude_destino.between(-180, 180)).sum())],
})
display(nulos)
display(faixas)
if nulos.nulos.sum() or faixas.linhas.sum():
    raise ValueError("Há dados inválidos; corrija a origem antes de avaliar o modelo")
print("Disponibilidade antes da corrida: distancia_lalo/coordenadas = sim; calendario derivado da entrega = não")

## 6. Validação cruzada do KNN e referência simples
O professor usa um KNN com cinco vizinhos e `MinMaxScaler`. Aqui aplicamos esse **mesmo pipeline** em cinco dobras do treino, comparando seu MAE com uma previsão constante pela média. O pipeline é ajustado dentro de cada dobra para não usar estatísticas da validação no treino. O teste de 20% fica reservado para uma única avaliação final.

Com 104 corridas, o resultado pode variar bastante entre divisões; a validação cruzada mostra essa variação, mas não resolve o vazamento da hora de entrega.

In [ ]:
X = ml.drop(columns=[ALVO]).copy()
y = ml[ALVO].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TESTE, random_state=SEMENTE
)

def pipeline_knn():
    preprocessador = ColumnTransformer(
        transformers=[("drop", "drop", ["periodo"]),
                      ("minmax", MinMaxScaler(), FEATURES)],
        remainder="passthrough"
    )
    return Pipeline(steps=[("preprocessamento", preprocessador),
                           ("knn", KNeighborsRegressor(n_neighbors=5))])

knn = pipeline_knn()
referencia = DummyRegressor(strategy="mean")
cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)
mae_knn = -cross_val_score(knn, X_train, y_train, cv=cv,
                           scoring="neg_mean_absolute_error", error_score="raise")
mae_referencia = -cross_val_score(referencia, X_train, y_train, cv=cv,
                                  scoring="neg_mean_absolute_error", error_score="raise")
display(pd.DataFrame({"dobra": range(1, 6), "MAE_KNN_R$": mae_knn,
                      "MAE_media_R$": mae_referencia}).round(3))
print(f"KNN: MAE CV médio R$ {mae_knn.mean():.2f} (desvio R$ {mae_knn.std(ddof=1):.2f})")
print(f"Referência pela média: MAE CV médio R$ {mae_referencia.mean():.2f}")

## 7. Avaliação única no teste reservado
MAE e RMSE são medidos em reais. A comparação com a média é apenas um controle acadêmico; ela **não substitui** o modelo atual da DESLOC.

In [ ]:
knn.fit(X_train, y_train)
referencia.fit(X_train, y_train)
pred_knn = knn.predict(X_test)
pred_referencia = referencia.predict(X_test)

def avaliar(nome, y_real, y_previsto):
    return {"modelo": nome, "corridas_teste": len(y_real),
            "MAE_R$": mean_absolute_error(y_real, y_previsto),
            "RMSE_R$": sqrt(mean_squared_error(y_real, y_previsto)),
            "vies_medio_R$": float(np.mean(np.asarray(y_previsto) - np.asarray(y_real)))}

resultado_teste = pd.DataFrame([
    avaliar("KNN_5", y_test, pred_knn),
    avaliar("media", y_test, pred_referencia),
])
display(resultado_teste.round(3))

## 8. XAI global — importância por permutação
**Objetivo:** identificar quais entradas alteram o erro do KNN quando seus valores são embaralhados no teste. A unidade abaixo é o aumento médio do MAE em reais; valor maior sugere maior dependência do modelo daquela entrada **nesta amostra**.

**Limites:** o teste tem cerca de 21 corridas; coordenadas e distância podem ser correlacionadas. Uma importância baixa isolada não prova que a feature é irrelevante. `periodo` foi removido pelo pipeline, portanto sua importância deve ser zero. Valores negativos podem ocorrer por ruído numa amostra pequena: embaralhar a feature reduziu o erro naquele teste. Este gráfico não mede causalidade, justiça ou desempenho futuro.

In [ ]:
importancias = permutation_importance(
    knn, X_test, y_test, scoring="neg_mean_absolute_error",
    n_repeats=10, random_state=SEMENTE
)
xai_global = pd.DataFrame({
    "feature": X_test.columns,
    "aumento_MAE_medio_R$": importancias.importances_mean,
    "desvio_R$": importancias.importances_std,
}).sort_values("aumento_MAE_medio_R$", ascending=False)
display(xai_global.round(3))
fig, ax = plt.subplots(figsize=(9, 5))
ordem = xai_global.sort_values("aumento_MAE_medio_R$")
ax.barh(ordem.feature, ordem["aumento_MAE_medio_R$"], color="#2878B5")
ax.axvline(0, color="black", lw=.8)
ax.set(title="XAI global do KNN no teste reservado",
       xlabel="Aumento médio do MAE ao permutar a feature (R$)")
plt.tight_layout(); plt.show()

## 9. XAI local — cinco corridas mais próximas
**Objetivo:** mostrar, para uma corrida do teste, quais cinco corridas do treino sustentam a previsão do KNN. O professor usa `n_neighbors=5` e pesos uniformes; a previsão é a média dos cinco preços vizinhos após a escala MinMax.

**Limites:** “próximo” significa próximo nas features escaladas, não necessariamente no mapa. Esta é uma explicação operacional do KNN, não uma afirmação causal sobre o preço.

In [ ]:
posicao_exemplo = 0
linha_teste = X_test.iloc[[posicao_exemplo]]
preprocessador_ajustado = knn.named_steps["preprocessamento"]
knn_ajustado = knn.named_steps["knn"]
linha_escalada = preprocessador_ajustado.transform(linha_teste)
distancias_vizinhos, posicoes_vizinhos = knn_ajustado.kneighbors(
    linha_escalada, n_neighbors=5
)
posicoes = posicoes_vizinhos[0]
vizinhos = pd.DataFrame({
    "ordem_vizinho": range(1, 6),
    "distancia_no_espaco_escalado": distancias_vizinhos[0],
    "preco_vizinho_R$": y_train.iloc[posicoes].to_numpy(),
    "distancia_geografica_km": X_train.iloc[posicoes].distancia_lalo.to_numpy(),
    "hora_entrega": X_train.iloc[posicoes].hora.to_numpy(),
})
display(vizinhos.round(3))
print(f"Preço real do exemplo: R$ {y_test.iloc[posicao_exemplo]:.2f}")
print(f"Preço previsto pelo KNN: R$ {pred_knn[posicao_exemplo]:.2f}")
print(f"Média dos cinco vizinhos: R$ {vizinhos['preco_vizinho_R$'].mean():.2f}")

## 10. Dados necessários para comparar o modelo atual e medir drift

**Previsões do modelo atual:** identificador da corrida, versão do modelo, horário da solicitação, entradas disponíveis nesse momento, preço previsto e valor efetivamente pago. Com as mesmas corridas, calcular MAE, RMSE, viés e erros por distância, período e região. Sem previsões registradas, não há comparação válida.

**Drift:** conjunto usado no treino original, data de referência, versão das features e uma janela recente posterior. Comparar nulos, faixas, medianas/distribuições e segmentos. A tabela Gold atual não guarda identificador da corrida nem timestamp completo como metadado; isso dificulta alinhamento temporal e auditoria.

**Impacto de negócio:** solicitações, aceite, cancelamento, reclamações e custos por corrida para medir conversão e margem. `SUM(vlr_pago)` é valor pago bruto, não lucro.

**Critério de execução:** enquanto essas fontes não existirem, registrar **PENDENTE POR AUSÊNCIA DE DADOS**. Não criar números simulados como se fossem medições da produção.

In [ ]:
pendencias = pd.DataFrame([
    {"evidencia": "Erro do modelo atual nas mesmas corridas", "status": "PENDENTE", "fonte_necessaria": "logs de previsão + preço realizado"},
    {"evidencia": "Drift em relação ao treino original", "status": "PENDENTE", "fonte_necessaria": "snapshot do treino + janela recente"},
    {"evidencia": "Features conhecidas antes da cotação", "status": "NÃO ATENDE", "fonte_necessaria": "datahora_solicitacao ou coleta"},
    {"evidencia": "Conversão, cancelamento e margem", "status": "PENDENTE", "fonte_necessaria": "eventos do app + custos"},
    {"evidencia": "MAE/RMSE do KNN nesta amostra", "status": "DIAGNÓSTICO", "fonte_necessaria": TABELA_ML},
])
display(pendencias)
print("Conclusão atual: não há evidência suficiente para decidir retreino ou implantação.")

## 11. Próximo experimento e decisão

1. **Corrigir** o pipeline para usar data/hora da solicitação ou coleta como entrada. Guardar `id_corrida` e timestamp como metadados para divisão temporal e rastreio, sem usá-los indevidamente como features.
2. **Registrar** entradas e previsões do modelo atual; unir ao valor pago depois da corrida pelo identificador.
3. **Separar** um período futuro de teste e avaliar o atual e o candidato nas mesmas corridas. Usar validação cruzada somente no treino para escolher hiperparâmetros; evitar escolher modelo pelo teste.
4. **Comparar** MAE/RMSE e erros por segmento com metas aprovadas. Investigar drift e impacto em conversão/margem.
5. **Decidir** manter, testar em modo sombra/A/B ou retreinar e substituir. Com os dados atuais, o status correto é **decisão pendente**.

Este notebook não grava tabelas, não registra modelo e não cria endpoint.